In [7]:
from pathlib import Path
import re, datetime as dt
import numpy as np, geopandas as gpd
import rioxarray as rxr
import matplotlib.pyplot as plt

SHP_ROOT = Path(r"D:\Masters\marine-debris.github.io\data\shapefiles")  # folder to search

def parse_scene_from_ref(date_dir: Path):
    """Return (ref_path, tile, date_iso) from S2 B02 filename S2_<TILE>_<YYYY-MM-DD>_B02.tif"""
    ref = next((date_dir/"optical").glob("S2_*_B02.tif"))
    m = re.search(r"S2_([0-9A-Z]{5})_(\d{4}-\d{2}-\d{2})_B02\.tif$", ref.name, re.I)
    if not m:
        raise ValueError(f"Unexpected S2 ref name: {ref.name}")
    tile, date_iso = m.group(1), m.group(2)
    return ref, tile.upper(), date_iso

def _iso_from_ddmmyy(d, m, y2):
    y = 2000 + int(y2)  # assume 20xx
    return dt.date(y, int(m), int(d)).isoformat()

def find_matching_shp(shp_root: Path, tile: str, date_iso: str):
    """Shapefiles look like: S2_<DD-M-YY>_<TILE>.shp (e.g., S2_04-9-16_16PCC.shp)."""
    cands = list(shp_root.glob(f"S2_*_{tile}.shp"))
    for p in cands:
        mm = re.search(r"S2_(\d{1,2})-(\d{1,2})-(\d{2})_"+re.escape(tile)+r"\.shp$", p.name, re.I)
        if not mm: 
            continue
        d,m,y = mm.groups()
        if _iso_from_ddmmyy(d,m,y) == date_iso:
            return p
    return None

def _stretch(img, p=(2,98)):
    lo, hi = np.nanpercentile(img, p)
    return np.clip((img - lo)/(hi - lo + 1e-6), 0, 1)

def quicklook_s2_with_shp(date_dir: Path, shp_path: Path):
    ref = rxr.open_rasterio(next((date_dir/"optical").glob("S2_*_B02.tif"))).squeeze(drop=True)
    gdf = gpd.read_file(shp_path)
    if "id" in gdf.columns:
        gdf = gdf[gdf["id"] == 1].copy()
    gdf = gdf.to_crs(ref.rio.crs)

    xmin, ymin, xmax, ymax = ref.rio.bounds()
    gdf_clip = gdf.cx[xmin:xmax, ymin:ymax]

    fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(_stretch(ref.values.astype("float32")),
              extent=[ref.x.min(), ref.x.max(), ref.y.min(), ref.y.max()],
              origin="upper", cmap="gray")
    if not gdf_clip.empty:
        # polygons/lines in red, points in yellow
        polyline = gdf_clip[gdf_clip.geometry.type.isin(
            ["Polygon","MultiPolygon","LineString","MultiLineString"])]
        points = gdf_clip[gdf_clip.geometry.type.isin(["Point","MultiPoint"])]
        if not polyline.empty:
            polyline.boundary.plot(ax=ax, edgecolor="red", linewidth=1)
        if not points.empty:
            points.plot(ax=ax, color="yellow", markersize=10)
    ax.set_title(f"{date_dir.name} — S2 B02 + debris (id==1)")
    ax.set_xlim(ref.x.min(), ref.x.max()); ax.set_ylim(ref.y.min(), ref.y.max())
    plt.show()


In [15]:
import re, datetime as dt
import rioxarray as rxr
from pathlib import Path

def parse_scene_from_ref(date_dir: Path):
    ref = next((date_dir/"optical").glob("S2_*_B02.tif"))
    m = re.search(r"S2_([0-9A-Z]{5})_(\d{4}-\d{2}-\d{2})_B02\.tif$", ref.name, re.I)
    if not m:
        raise ValueError(f"Unexpected S2 B02 name: {ref.name}")
    return ref, m.group(1).upper(), m.group(2)

def iso_from_ddmmyy(d, m, y2):  # S2_DD-M-YY_TILE.shp
    return dt.date(2000+int(y2), int(m), int(d)).isoformat()

def find_matching_shp(shp_root: Path, tile: str, date_iso: str, max_days_tol=1):
    candidates = []
    for p in shp_root.glob(f"S2_*_{tile}.shp"):
        m = re.match(rf"S2_(\d{{1,2}})-(\d{{1,2}})-(\d{{2}})_{re.escape(tile)}\.shp$", p.name, re.I)
        if not m: 
            continue
        iso = iso_from_ddmmyy(*m.groups())
        candidates.append((iso, p))
        if iso == date_iso:
            return p
    if not candidates:
        raise FileNotFoundError(f"No shapefiles for tile {tile} in {shp_root}")
    # use nearest date within tolerance if exact not found
    d0 = dt.date.fromisoformat(date_iso)
    candidates.sort(key=lambda t: abs((dt.date.fromisoformat(t[0]) - d0).days))
    best_iso, best = candidates[0]
    if abs((dt.date.fromisoformat(best_iso) - d0).days) <= max_days_tol:
        print(f"⚠ Using nearest shapefile date {best_iso}: {best.name}")
        return best
    raise FileNotFoundError(f"No shapefile within {max_days_tol} day(s) of {date_iso} for tile {tile}")

# resolve ref + shapefile
ref_tif, TILE, DATE_ISO = parse_scene_from_ref(DATE_DIR)
SHP_PATH = find_matching_shp(SHP_ROOT, TILE, DATE_ISO)
print("Tile/date:", TILE, DATE_ISO)
print("Shapefile:", SHP_PATH)


Tile/date: 16PCC 2016-09-04
Shapefile: D:\Masters\marine-debris.github.io\data\shapefiles\S2_4-9-16_16PCC.shp


In [24]:
# === OpenDrift Plastics for one scene (auto shapefile match, fixed final extraction) ===
from pathlib import Path
import os, re, time, numpy as np
import geopandas as gpd
import xarray as xr, rioxarray as rxr
from pyproj import Transformer
from shapely.geometry import box
from datetime import timedelta
import pandas as pd

from opendrift.models.plastdrift import PlastDrift
from opendrift.readers import reader_netCDF_CF_generic

# ---------- ROOTS ----------
DRIFT_ROOT = Path(r"D:\Masters\Drift")
SHP_ROOT   = Path(r"D:\Masters\marine-debris.github.io\data\shapefiles")

# ---------- knobs ----------
DECIMATE     = 30            # grid thinning for speed
ENSEMBLE_N   = 20            # members per seed
DT_MIN       = 60            # minutes per time step
WINDAGE      = 0.03          # 3% windage for macroplastics
WINDAGE_STD  = 0.01          # ± spread per member
TERM_VEL     = 0.01          # m/s buoyant rise
BOX_HALF_M   = 500           # 1-km square (half side)
START_TIME0  = np.datetime64("2000-01-01T00:00:00")

# ---------- small helpers ----------
def _open_match(p: Path, ref):
    da = rxr.open_rasterio(p).squeeze(drop=True)
    return da.rio.reproject_match(ref)

def _dec(v, step):  # decimate for speed
    return v.isel(y=slice(None,None,step), x=slice(None,None,step)).astype("float32")

def _atomic_netcdf_write(ds: xr.Dataset, out_nc: Path):
    tmp = out_nc.with_suffix(out_nc.suffix + f".tmp.{int(time.time())}")
    ds.to_netcdf(tmp, mode="w", engine="netcdf4")
    ds.close()
    if out_nc.exists():
        try:
            os.remove(out_nc)
        except PermissionError:
            time.sleep(0.5)
            try: os.remove(out_nc)
            except Exception: pass
    os.replace(str(tmp), str(out_nc))

def get_optical_ref(date_dir: Path) -> Path | None:
    opt = date_dir/"optical"
    return next(opt.glob("S2_*_B02.tif"), None)

def parse_tile_date_from_optical(ref_tif: Path):
    m = re.match(r"S2_([A-Z0-9]{5})_(\d{4}-\d{2}-\d{2})_B02\.tif$", ref_tif.name)
    if not m:
        raise ValueError(f"Unexpected optical name: {ref_tif.name}")
    tile = m.group(1)
    date = pd.to_datetime(m.group(2)).date()
    return tile, date

def candidate_shp_names(date, tile):
    yy = f"{date.year%100:02d}"
    dZ, dN = f"{date.day:02d}", str(date.day)
    mZ, mN = f"{date.month:02d}", str(date.month)
    stems = [
        f"S2_{dN}-{mN}-{yy}_{tile}",
        f"S2_{dZ}-{mN}-{yy}_{tile}",
        f"S2_{dN}-{mZ}-{yy}_{tile}",
        f"S2_{dZ}-{mZ}-{yy}_{tile}",
    ]
    return [s + ".shp" for s in stems]

def find_matching_shapefile(shp_root: Path, tile, date) -> Path | None:
    for name in candidate_shp_names(date, tile):
        p = shp_root / name
        if p.exists():
            return p
    patt = re.compile(r"S2_(\d{1,2})-(\d{1,2})-(\d{2})_([A-Z0-9]{5})\.shp$")
    for p in shp_root.glob("S2_*-*-*_*.shp"):
        m = patt.match(p.name)
        if not m: continue
        d, mth, yy2, tl = map(str, m.groups())
        if tl != tile: continue
        try:
            dt = pd.to_datetime(f"20{yy2}-{int(mth):02d}-{int(d):02d}").date()
        except Exception:
            continue
        if dt == date:
            return p
    return None

# ---------- forcing builder ----------
def build_plast_forcing(date_dir: Path, sar_dir: Path, decimate: int = DECIMATE) -> Path:
    ref = rxr.open_rasterio(get_optical_ref(date_dir)).squeeze(drop=True)
    bio0, bio1 = date_dir/"bio_s2", sar_dir/"bio"

    # --- S2-time
    uo0, vo0    = _open_match(bio0/"uo.tif", ref),   _open_match(bio0/"vo.tif", ref)
    vsdx0,vsdy0 = _open_match(bio0/"vsdx.tif", ref), _open_match(bio0/"vsdy.tif", ref)
    swh0 = _open_match(bio0/"swh.tif", ref) if (bio0/"swh.tif").exists() else xr.zeros_like(uo0)
    if (bio0/"u10.tif").exists() and (bio0/"v10.tif").exists():
        u10_0, v10_0 = _open_match(bio0/"u10.tif", ref), _open_match(bio0/"v10.tif", ref)
    else:
        wspd0, wdir0 = _open_match(bio0/"wind.tif", ref), _open_match(bio0/"wind_dir.tif", ref)
        th0 = np.deg2rad(wdir0)
        u10_0 = (wspd0*np.sin(th0)).astype("float32")
        v10_0 = (wspd0*np.cos(th0)).astype("float32")

    # --- SAR-time
    uo1, vo1    = _open_match(bio1/"uo.tif", ref),   _open_match(bio1/"vo.tif", ref)
    vsdx1,vsdy1 = _open_match(bio1/"vsdx.tif", ref), _open_match(bio1/"vsdy.tif", ref)
    swh1 = _open_match(bio1/"swh.tif", ref) if (bio1/"swh.tif").exists() else xr.zeros_like(uo1)
    if (bio1/"u10.tif").exists() and (bio1/"v10.tif").exists():
        u10_1, v10_1 = _open_match(bio1/"u10.tif", ref), _open_match(bio1/"v10.tif", ref)
    else:
        wspd1, wdir1 = _open_match(bio1/"wind.tif", ref), _open_match(bio1/"wind_dir.tif", ref)
        th1 = np.deg2rad(wdir1)
        u10_1 = (wspd1*np.sin(th1)).astype("float32")
        v10_1 = (wspd1*np.cos(th1)).astype("float32")

    # decimate
    uo0,vo0,vsdx0,vsdy0,u10_0,v10_0,swh0 = map(lambda d:_dec(d, decimate), (uo0,vo0,vsdx0,vsdy0,u10_0,v10_0,swh0))
    uo1,vo1,vsdx1,vsdy1,u10_1,v10_1,swh1 = map(lambda d:_dec(d, decimate), (uo1,vo1,vsdx1,vsdy1,u10_1,v10_1,swh1))

    # lat/lon
    xs, ys = uo0.x.values, uo0.y.values
    X, Y = np.meshgrid(xs, ys)
    to_wgs = Transformer.from_crs(ref.rio.crs, 4326, always_xy=True)
    lon, lat = to_wgs.transform(X, Y)

    # time from Δt folder
    m = re.search(r"SAR_([+-]?\d+\.?\d*)h", sar_dir.name); assert m, f"Bad SAR dir: {sar_dir}"
    delta_h = float(m.group(1))
    times = np.array([START_TIME0, START_TIME0 + np.timedelta64(int(abs(delta_h)*3600), 's')], dtype='datetime64[ns]')

    ds = xr.Dataset(
        coords=dict(
            time=("time", times),
            y=("y", np.arange(lon.shape[0], dtype=np.int32)),
            x=("x", np.arange(lon.shape[1], dtype=np.int32)),
            latitude =(("y","x"), lat.astype("float32")),
            longitude=(("y","x"), lon.astype("float32")),
        ),
        data_vars=dict(
            uo   =(("time","y","x"), np.stack([uo0.values,   uo1.values]).astype("float32")),
            vo   =(("time","y","x"), np.stack([vo0.values,   vo1.values]).astype("float32")),
            vsdx =(("time","y","x"), np.stack([vsdx0.values, vsdx1.values]).astype("float32")),
            vsdy =(("time","y","x"), np.stack([vsdy0.values, vsdy1.values]).astype("float32")),
            u10  =(("time","y","x"), np.stack([u10_0.values, u10_1.values]).astype("float32")),
            v10  =(("time","y","x"), np.stack([v10_0.values, v10_1.values]).astype("float32")),
            swh  =(("time","y","x"), np.stack([swh0.values,  swh1.values]).astype("float32")),
        )
    )
    # CF attrs
    ds["uo"].attrs.update(  standard_name="eastward_sea_water_velocity",  units="m s-1")
    ds["vo"].attrs.update(  standard_name="northward_sea_water_velocity", units="m s-1")
    ds["vsdx"].attrs.update(standard_name="sea_surface_wave_stokes_drift_x_velocity", units="m s-1")
    ds["vsdy"].attrs.update(standard_name="sea_surface_wave_stokes_drift_y_velocity", units="m s-1")
    ds["u10"].attrs.update( standard_name="eastward_wind",  units="m s-1")
    ds["v10"].attrs.update( standard_name="northward_wind", units="m s-1")
    ds["swh"].attrs.update( standard_name="sea_surface_wave_significant_height", units="m")

    out_nc = sar_dir/"forcing_plast.nc"
    _atomic_netcdf_write(ds, out_nc)
    print("Wrote", out_nc)
    return out_nc

def seeds_wgs84_from_shp(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.get("id", 0) == 1].copy()
    if gdf.empty:
        return [], []
    reps = []
    for g in gdf.geometry:
        if g.geom_type == "Point": reps.append(g)
        elif g.geom_type in ("Polygon","MultiPolygon"): reps.append(g.representative_point())
        else: reps.append(g.centroid)
    g2 = gpd.GeoDataFrame(geometry=reps, crs=gdf.crs).to_crs(4326)
    return [p.x for p in g2.geometry], [p.y for p in g2.geometry]

def extract_final_lonlat(o: PlastDrift):
    """Robust: prefer new API (o.result), fallback to get_property; return 1-D arrays without NaNs."""
    try:
        lon = o.result['lon'].isel(time=-1).values
        lat = o.result['lat'].isel(time=-1).values
    except Exception:
        lon = np.asarray(o.get_property('lon')[-1])
        lat = np.asarray(o.get_property('lat')[-1])
    lon = np.asarray(lon).ravel()
    lat = np.asarray(lat).ravel()
    m = np.isfinite(lon) & np.isfinite(lat)
    return lon[m], lat[m]

# ---------- run one scene ----------
def run_plast(date_dir: Path, sar_dir: Path, shp_path: Path):
    if not shp_path.exists():
        raise FileNotFoundError(f"Shapefile not found: {shp_path}")

    nc = sar_dir/"forcing_plast.nc"
    if not nc.exists():
        nc = build_plast_forcing(date_dir, sar_dir, decimate=DECIMATE)
    rdr = reader_netCDF_CF_generic.Reader(nc)

    o = PlastDrift(loglevel=20)
    o.set_config('general:use_auto_landmask', True)            # add GSHHG landmask
    o.set_config('drift:use_tabularised_stokes_drift', False)  # use VSDX/VSDY provided
    o.set_config('seed:wind_drift_factor', WINDAGE)
    o.add_reader(rdr)

    # seeds
    lons, lats = seeds_wgs84_from_shp(shp_path)
    if not lons:
        raise RuntimeError("No id==1 features in shapefile / within scene.")
    lons_e = np.repeat(lons, ENSEMBLE_N); lats_e = np.repeat(lats, ENSEMBLE_N)

    seed_time = pd.Timestamp(START_TIME0).to_pydatetime()
    o.seed_elements(lon=lons_e.tolist(), lat=lats_e.tolist(),
                    number=len(lons_e), time=seed_time,
                    z=0.0, terminal_velocity=TERM_VEL)

    # jitter per element (use actual element count)
    n_el = o.elements.lon.size
    if WINDAGE_STD > 0 and n_el > 0:
        o.elements.wind_drift_factor = (
            o.elements.wind_drift_factor + np.random.normal(0, WINDAGE_STD, size=n_el)
        )

    # Δt from SAR folder
    m = re.search(r"SAR_([+-]?\d+\.?\d*)h", sar_dir.name); assert m, f"Bad SAR dir: {sar_dir.name}"
    delta_h = float(m.group(1))
    end_time = (pd.Timestamp(START_TIME0) + pd.Timedelta(hours=abs(delta_h))).to_pydatetime()
    step = timedelta(minutes=DT_MIN)
    time_step = (step if delta_h >= 0 else -step)

    o.run(end_time=end_time, time_step=time_step, export_variables=[])

    # outputs (WGS84)
    lonf, latf = extract_final_lonlat(o)
    pred = gpd.GeoDataFrame({"obj_id": np.arange(lonf.size, dtype=int)},
                            geometry=gpd.points_from_xy(lonf, latf), crs=4326)
    pred.to_file(sar_dir/"predicted_points_plast.shp")

    # 1-km box around ensemble mean for each original seed
    ref = rxr.open_rasterio(get_optical_ref(date_dir)).squeeze(drop=True)
    to_scene = Transformer.from_crs(4326, ref.rio.crs, always_xy=True)
    back_wgs = Transformer.from_crs(ref.rio.crs, 4326, always_xy=True)

    L = len(lons)
    means = []
    for i in range(L):
        seg_lon = lonf[i*ENSEMBLE_N:(i+1)*ENSEMBLE_N]
        seg_lat = latf[i*ENSEMBLE_N:(i+1)*ENSEMBLE_N]
        if seg_lon.size == 0:
            means.append((np.nan, np.nan))
        else:
            means.append((float(np.nanmean(seg_lon)), float(np.nanmean(seg_lat))))

    boxes = []
    for lo, la in means:
        if not np.isfinite(lo) or not np.isfinite(la):
            continue
        x, y = to_scene.transform(lo, la)
        b = box(x-BOX_HALF_M, y-BOX_HALF_M, x+BOX_HALF_M, y+BOX_HALF_M)
        xs, ys = zip(*list(b.exterior.coords))
        lox, lay = back_wgs.transform(xs, ys)
        boxes.append(box(min(lox), min(lay), max(lox), max(lay)))

    box_gdf = gpd.GeoDataFrame({"seed_id": np.arange(len(boxes), dtype=int)}, geometry=boxes, crs=4326)
    box_gdf.to_file(sar_dir/"search_boxes_1km_plast.shp")

    print(f"✓ {sar_dir.name}: predicted_points_plast.shp, search_boxes_1km_plast.shp")

# ---------- choose one date folder and a SAR_±h folder ----------
DATE_DIR = DRIFT_ROOT / "2016-09-04"
opt_ref = get_optical_ref(DATE_DIR)
TILE, DATE = parse_tile_date_from_optical(opt_ref)
print(f"Optical ref: {opt_ref.name}  →  tile={TILE}, date={DATE}")

sar_dirs = sorted(DATE_DIR.glob("SAR_*h"))
SAR_DIR = sar_dirs[0]
print(f"Using SAR folder: {SAR_DIR.name}")

SHP_PATH = find_matching_shapefile(SHP_ROOT, TILE, DATE)
print(f"Using shapefile: {SHP_PATH.name}")

# ---------- RUN ----------
run_plast(DATE_DIR, SAR_DIR, SHP_PATH)


15:21:51 INFO    opendrift.readers:61: Opening file with xr.open_dataset
15:21:51 INFO    opendrift.readers.reader_netCDF_CF_generic:332: Detected dimensions: {'time': 'time', 'x': 'y', 'y': 'x'}
15:21:51 WARNING opendrift.readers.basereader.structured:50: No proj string or projection could be derived, using 'fakeproj'. This assumes that the variables are structured and gridded approximately equidistantly on the surface (i.e. in meters). This must be guaranteed by the user. You can get rid of this warning by supplying a valid projection to the reader.
15:21:51 INFO    opendrift.readers.basereader.structured:90: Making interpolator for lon,lat to x,y conversion...


Optical ref: S2_16PCC_2016-09-04_B02.tif  →  tile=16PCC, date=2016-09-04
Using SAR folder: SAR_+7.9h
Using shapefile: S2_4-9-16_16PCC.shp


15:21:53 INFO    opendrift.readers.basereader:176: Variable x_sea_water_velocity will be rotated from eastward_sea_water_velocity
15:21:53 INFO    opendrift.readers.basereader:176: Variable y_sea_water_velocity will be rotated from northward_sea_water_velocity
15:21:53 INFO    opendrift.readers.basereader:176: Variable x_wind will be rotated from eastward_wind
15:21:53 INFO    opendrift.readers.basereader:176: Variable y_wind will be rotated from northward_wind
15:21:53 INFO    opendrift:509: OpenDriftSimulation initialised (version 1.14.2)
15:21:53 INFO    opendrift.models.basemodel.environment:206: Adding a global landmask from GSHHG
15:21:53 INFO    opendrift.models.basemodel.environment:229: Fallback values will be used for the following variables which have no readers: 
15:21:53 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_height: 0.000000
15:21:53 INFO    opendrift.models.basemodel.environment:232: 	ocean_vertical_diffusivity: 0.020000
15:21:53 INFO    opendri

✓ SAR_+7.9h: predicted_points_plast.shp, search_boxes_1km_plast.shp


In [25]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rioxarray as rxr
from pathlib import Path

def quicklook_result(date_dir: Path, sar_dir: Path, shp_path: Path, out_png: Path|None=None):
    ref = rxr.open_rasterio(next((date_dir/"optical").glob("S2_*_B02.tif"))).squeeze(drop=True)
    extent = [ref.x.min(), ref.x.max(), ref.y.min(), ref.y.max()]
    crs = ref.rio.crs

    # seeds (id==1), projected to scene CRS
    gdf0 = gpd.read_file(shp_path)
    if "id" in gdf0.columns:
        gdf0 = gdf0[gdf0["id"] == 1]
    seeds = gdf0.to_crs(crs)

    # results, reproject to scene CRS
    preds = gpd.read_file(sar_dir/"predicted_points_plast.shp").to_crs(crs)
    boxes = gpd.read_file(sar_dir/"search_boxes_1km_plast.shp").to_crs(crs)

    fig, ax = plt.subplots(figsize=(7,7))
    ax.imshow(ref.values, extent=extent, origin="upper", cmap="gray")
    seeds.plot(ax=ax, markersize=10, color="deepskyblue", label="Seeds (id=1)")
    preds.plot(ax=ax, markersize=10, color="crimson", label="Predicted points")
    boxes.boundary.plot(ax=ax, linewidth=1.2, color="orange", label="1 km boxes")
    ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
    ax.set_title(f"{date_dir.name} • {sar_dir.name}")
    ax.legend(loc="upper right")
    ax.set_axis_off()
    if out_png is None:
        out_png = sar_dir / "quicklook.png"
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("Saved", out_png)

# run for your example
from pathlib import Path
DATE_DIR = Path(r"D:\Masters\Drift\2016-09-04")
SAR_DIR  = DATE_DIR / "SAR_+7.9h"
SHP_PATH = Path(r"D:\Masters\marine-debris.github.io\data\shapefiles") / "S2_4-9-16_16PCC.shp"

quicklook_result(DATE_DIR, SAR_DIR, SHP_PATH)


15:23:33 WARNING py.warnings:112: C:\Users\Joshua Pretorius\AppData\Local\Temp\ipykernel_15676\2167939298.py:28: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend(loc="upper right")



Saved D:\Masters\Drift\2016-09-04\SAR_+7.9h\quicklook.png


In [28]:
# --- FIXED animation helper (no reliance on OpenDrift time array) ---
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.pyplot as plt
from pyproj import Transformer
import numpy as np
from datetime import timedelta
import re

SUBSAMPLE   = 4
POINT_SIZE  = 6
GIF_DPI     = 160

def animate_plast(date_dir: Path, sar_dir: Path, shp_path: Path, out_gif: Path|None=None):
    # 1) basemap
    ref = rxr.open_rasterio(get_optical_ref(date_dir)).squeeze(drop=True)
    extent = [ref.x.min(), ref.x.max(), ref.y.min(), ref.y.max()]
    to_scene = Transformer.from_crs(4326, ref.rio.crs, always_xy=True)

    # 2) reader/model (same settings as your run)
    nc = sar_dir / "forcing_plast.nc"
    if not nc.exists():
        nc = build_plast_forcing(date_dir, sar_dir, decimate=DECIMATE)
    rdr = reader_netCDF_CF_generic.Reader(nc)

    o = PlastDrift(loglevel=20)
    o.add_reader(rdr)
    o.set_config('general:use_auto_landmask', True)
    o.set_config('drift:use_tabularised_stokes_drift', False)
    o.set_config('seed:wind_drift_factor', WINDAGE)

    lons, lats = seeds_wgs84_from_shp(shp_path)
    if not lons:
        raise RuntimeError("No id==1 features in shapefile/scene.")
    lons_e = np.repeat(lons, ENSEMBLE_N)
    lats_e = np.repeat(lats, ENSEMBLE_N)
    o.seed_elements(lon=lons_e.tolist(), lat=lats_e.tolist(),
                    number=len(lons_e),
                    time=pd.Timestamp(START_TIME0).to_pydatetime(),
                    z=0.0, terminal_velocity=TERM_VEL)
    if WINDAGE_STD > 0:
        o.elements.wind_drift_factor = (
            o.elements.wind_drift_factor + np.random.normal(0, WINDAGE_STD, size=o.elements.lon.size)
        )

    m = re.search(r"SAR_([+-]?\d+\.?\d*)h", sar_dir.name); assert m, f"Bad SAR dir: {sar_dir.name}"
    delta_h = float(m.group(1))
    end_time = (pd.Timestamp(START_TIME0) + pd.Timedelta(hours=abs(delta_h))).to_pydatetime()
    step = timedelta(minutes=DT_MIN)
    o.run(end_time=end_time, time_step=(step if delta_h >= 0 else -step), export_variables=[])

    # 3) collect trajectories
    try:
        LON = np.asarray(o.result['lon'])
        LAT = np.asarray(o.result['lat'])
    except Exception:
        LON = np.asarray(o.get_property('lon'))
        LAT = np.asarray(o.get_property('lat'))

    # subsample elements for speed
    idx = np.arange(LON.shape[1])[::max(1, SUBSAMPLE)]
    LONs, LATs = LON[:, idx], LAT[:, idx]
    nframes = LONs.shape[0]
    sign = 1 if delta_h >= 0 else -1
    rel_minutes = sign * np.arange(nframes) * DT_MIN

    # 4) animate
    fig, ax = plt.subplots(figsize=(7,7))
    ax.imshow(ref.values, extent=extent, origin='upper', cmap='gray')
    ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
    ax.set_axis_off()

    # first frame
    x0, y0 = to_scene.transform(LONs[0], LATs[0])
    mask = np.isfinite(x0) & np.isfinite(y0)
    scat = ax.scatter(np.array(x0)[mask], np.array(y0)[mask], s=POINT_SIZE, alpha=0.85)

    title = ax.set_title(f"{date_dir.name} • {sar_dir.name}  (t={rel_minutes[0]:+d} min)")

    def update(i):
        xi, yi = to_scene.transform(LONs[i], LATs[i])
        mask = np.isfinite(xi) & np.isfinite(yi)
        scat.set_offsets(np.c_[np.array(xi)[mask], np.array(yi)[mask]])
        title.set_text(f"{date_dir.name} • {sar_dir.name}  (t={rel_minutes[i]:+d} min)")
        return scat, title

    ani = FuncAnimation(fig, update, frames=nframes, interval=200, blit=True)

    if out_gif is None:
        out_gif = sar_dir / "drift_animation_plast.gif"
    ani.save(out_gif, writer=PillowWriter(fps=max(2, int(60/DT_MIN))), dpi=GIF_DPI)
    plt.close(fig)
    print("Saved animation:", out_gif)


In [ ]:
animate_plast(DATE_DIR, SAR_DIR, SHP_PATH)

15:43:54 INFO    opendrift.readers:61: Opening file with xr.open_dataset
15:43:54 INFO    opendrift.readers.reader_netCDF_CF_generic:332: Detected dimensions: {'time': 'time', 'x': 'y', 'y': 'x'}
15:43:54 WARNING opendrift.readers.basereader.structured:50: No proj string or projection could be derived, using 'fakeproj'. This assumes that the variables are structured and gridded approximately equidistantly on the surface (i.e. in meters). This must be guaranteed by the user. You can get rid of this warning by supplying a valid projection to the reader.
15:43:54 INFO    opendrift.readers.basereader.structured:90: Making interpolator for lon,lat to x,y conversion...
15:43:55 INFO    opendrift.readers.basereader:176: Variable x_sea_water_velocity will be rotated from eastward_sea_water_velocity
15:43:55 INFO    opendrift.readers.basereader:176: Variable y_sea_water_velocity will be rotated from northward_sea_water_velocity
15:43:55 INFO    opendrift.readers.basereader:176: Variable x_wind 